# Deep Learning

**Case study:** Customer spending prediction using neural networks. Synthetic data only. Run cells in order.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path("data") if Path("data/customers_synthetic.csv").exists() else Path(".")
customers = pd.read_csv(DATA / "customers_synthetic.csv")
orders = pd.read_csv(DATA / "orders_synthetic.csv", parse_dates=["order_date"])
print("Customers:", customers.shape, "Orders:", orders.shape)
display(customers.head())

## Data quality and exploratory checks

In [ ]:
assert customers.customer_id.is_unique
assert orders.order_id.is_unique
assert customers.churned.isin([0,1]).all()
print(customers.isna().sum().to_string())
print("Churn rate:", round(customers.churned.mean(),3))
customers.groupby("plan").churned.agg(["mean","count"]).plot(kind="bar",subplots=True,figsize=(9,3),title=["Churn rate","Customers"]); plt.tight_layout(); plt.show()

## Prepare leakage-safe regression data and a linear baseline

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
features=["age","tenure_months","support_tickets","discount_pct","engagement_score"]
X=customers[features].to_numpy(dtype=float); y=customers.monthly_spend.to_numpy()
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42)
scaler=StandardScaler(); Xtr=scaler.fit_transform(Xtr); Xte=scaler.transform(Xte)
baseline=Ridge().fit(Xtr,ytr)
print("Ridge MAE:",round(mean_absolute_error(yte,baseline.predict(Xte)),2))

## Train a small neural network with early stopping

In [ ]:
import tensorflow as tf
tf.keras.utils.set_random_seed(42)
net=tf.keras.Sequential([tf.keras.layers.Input(shape=(Xtr.shape[1],)),tf.keras.layers.Dense(24,activation="relu"),tf.keras.layers.Dense(12,activation="relu"),tf.keras.layers.Dense(1)])
net.compile(optimizer="adam",loss="mse",metrics=["mae"])
early=tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=8,restore_best_weights=True)
history=net.fit(Xtr,ytr,validation_split=.2,epochs=60,batch_size=32,callbacks=[early],verbose=0)
pred=net.predict(Xte,verbose=0).ravel()
print("Neural net test MAE:",round(mean_absolute_error(yte,pred),2))
plt.plot(history.history["loss"],label="train");plt.plot(history.history["val_loss"],label="validation");plt.legend();plt.title("Training vs validation loss");plt.xlabel("Epoch");plt.show()

## Reflection

1. What was your baseline? 2. What could go wrong in deployment? 3. Which results require human review?

Complete `Practice_Questions.md` before reading `Solutions.md`.